# SignalCare AI — Utterance & Signal Distribution Analysis

This notebook provides a full exploratory analysis of the preprocessed HOPE_WSDM_2022 Train split.

**Inputs:**
- `processed/utterances.jsonl` — utterance-level data from `preprocess.py`
- `src/train_risk_signals.csv` — transcript-level risk signal labels from `risk_taxonomy.py`

**Sections:**
1. Load & validate data
2. Utterance-level stats (counts, lengths, turns per transcript)
3. Transcript-level signal distribution
4. Signal co-occurrence heatmap
5. Transcripts with no signals (null set analysis)
6. Top-K most distressed transcripts
7. Signal severity grouping

In [ ]:
import json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from collections import Counter
from itertools import combinations

# Plot styling
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.size'] = 11

# Paths
UTTERANCES_JSONL = r'C:\Users\samee\Documents\CHIEAC\processed\utterances.jsonl'
TRANSCRIPT_CSV   = r'C:\Users\samee\Documents\CHIEAC\src\train_risk_signals.csv'

print('Libraries loaded.')

## 1. Load & Validate Data

In [ ]:
# --- Load utterances ---
utterances = []
with open(UTTERANCES_JSONL, 'r', encoding='utf-8') as f:
    for line in f:
        utterances.append(json.loads(line.strip()))

utt_df = pd.DataFrame(utterances)
utt_df['text_length'] = utt_df['text'].str.split().str.len()

print(f'Utterances loaded : {len(utt_df)}')
print(f'Transcripts       : {utt_df["transcript_id"].nunique()}')
utt_df.head(3)

In [ ]:
# --- Load transcript-level signals ---
tx_df = pd.read_csv(TRANSCRIPT_CSV)

# Parse risk_signals column into list
def parse_signals(val):
    if pd.isna(val) or val == '':
        return []
    return [s.strip() for s in val.split(',') if s.strip()]

tx_df['signal_list'] = tx_df['risk_signals'].apply(parse_signals)
tx_df['signal_count'] = tx_df['signal_list'].apply(len)

print(f'Transcripts loaded : {len(tx_df)}')
print(f'Columns            : {list(tx_df.columns)}')
tx_df[['transcript_id', 'signal_list', 'signal_count']].head(5)

## 2. Utterance-Level Stats

In [ ]:
# Turns per transcript
turns_per_tx = utt_df.groupby('transcript_id')['patient_turn_id'].max().reset_index()
turns_per_tx.columns = ['transcript_id', 'num_turns']

print('=== Turns per Transcript ===')
print(turns_per_tx['num_turns'].describe().round(2))

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Distribution of turns per transcript
axes[0].hist(turns_per_tx['num_turns'], bins=20, color='steelblue', edgecolor='white')
axes[0].set_title('Distribution of Patient Turns per Transcript')
axes[0].set_xlabel('Number of Patient Turns')
axes[0].set_ylabel('Number of Transcripts')

# Distribution of utterance word length
axes[1].hist(utt_df['text_length'], bins=40, color='darkorange', edgecolor='white')
axes[1].set_title('Distribution of Utterance Word Length')
axes[1].set_xlabel('Word Count per Utterance')
axes[1].set_ylabel('Frequency')

plt.tight_layout()
plt.show()

print(f'\nUtterance length stats (words):')
print(utt_df['text_length'].describe().round(2))

## 3. Transcript-Level Signal Distribution

In [ ]:
# Flatten all signals across all transcripts
all_signals = [sig for signals in tx_df['signal_list'] for sig in signals]
signal_counts = Counter(all_signals)

signal_df = pd.DataFrame(signal_counts.items(), columns=['signal', 'count'])
signal_df = signal_df.sort_values('count', ascending=False).reset_index(drop=True)

# Add percentage of transcripts
signal_df['pct_transcripts'] = (signal_df['count'] / len(tx_df) * 100).round(1)

print('=== Signal Frequency Across Transcripts ===')
print(signal_df.to_string(index=False))

In [ ]:
# Bar chart of signal frequency
plt.figure(figsize=(14, 6))
bars = plt.barh(
    signal_df['signal'][::-1],
    signal_df['count'][::-1],
    color='steelblue',
    edgecolor='white'
)
plt.xlabel('Number of Transcripts')
plt.title('Risk Signal Frequency (Transcript-Level Labels)')
plt.axvline(x=len(tx_df) * 0.1, color='red', linestyle='--', alpha=0.7, label='10% threshold')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Distribution of how many signals per transcript
plt.figure(figsize=(10, 5))
tx_df['signal_count'].value_counts().sort_index().plot(
    kind='bar', color='darkorange', edgecolor='white'
)
plt.title('Distribution of Signal Count per Transcript')
plt.xlabel('Number of Signals Detected')
plt.ylabel('Number of Transcripts')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

print(f"\nTranscripts with 0 signals : {(tx_df['signal_count'] == 0).sum()}")
print(f"Transcripts with 1+ signals: {(tx_df['signal_count'] > 0).sum()}")
print(f"Max signals on one transcript: {tx_df['signal_count'].max()}")

## 4. Signal Co-occurrence Heatmap

In [ ]:
# Build binary signal matrix (one row per transcript)
all_signal_names = sorted(set(all_signals))

binary_rows = []
for _, row in tx_df.iterrows():
    binary_row = {sig: int(sig in row['signal_list']) for sig in all_signal_names}
    binary_rows.append(binary_row)

binary_df = pd.DataFrame(binary_rows)

# Co-occurrence matrix: how often do signal A and signal B appear together?
cooccurrence = binary_df.T.dot(binary_df)

# Normalize by diagonal (Jaccard-style: co-occur / union)
diag = np.diag(cooccurrence.values)
union = diag[:, None] + diag[None, :] - cooccurrence.values
jaccard_matrix = pd.DataFrame(
    np.where(union == 0, 0, cooccurrence.values / union),
    index=all_signal_names,
    columns=all_signal_names
)

plt.figure(figsize=(14, 11))
mask = np.eye(len(all_signal_names), dtype=bool)  # hide diagonal
sns.heatmap(
    jaccard_matrix,
    mask=mask,
    annot=True,
    fmt='.2f',
    cmap='YlOrRd',
    linewidths=0.5,
    square=True,
    cbar_kws={'label': 'Jaccard Similarity'}
)
plt.title('Signal Co-occurrence Heatmap (Jaccard Similarity)')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.yticks(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Top 10 most co-occurring signal pairs
pairs = []
for s1, s2 in combinations(all_signal_names, 2):
    count = int(cooccurrence.loc[s1, s2])
    if count > 0:
        pairs.append({'signal_1': s1, 'signal_2': s2, 'co_count': count})

pairs_df = pd.DataFrame(pairs).sort_values('co_count', ascending=False).head(10)
print('=== Top 10 Most Co-occurring Signal Pairs ===')
print(pairs_df.to_string(index=False))

## 5. Null Set Analysis (Transcripts with No Signals)

In [ ]:
null_tx = tx_df[tx_df['signal_count'] == 0][['transcript_id', 'text']].copy()
null_tx['text_preview'] = null_tx['text'].str[:300]

print(f'Transcripts with no signals: {len(null_tx)}')
print('\nSample previews:')
for _, row in null_tx.head(5).iterrows():
    print(f"\n--- {row['transcript_id']} ---")
    print(row['text_preview'])

## 6. Top-K Most Distressed Transcripts

In [ ]:
# Weight signals by severity
# High severity: S01, S12, S13, S15, S17, S18
# Medium severity: S02, S04, S05, S07, S08, S11, S16
# Low severity: S03, S06, S09, S10, S14
SEVERITY_WEIGHTS = {
    'S01_hopelessness_escalation'    : 3,
    'S12_escalation_risk_composite'  : 3,
    'S13_suicidal_ideation'          : 3,
    'S15_self_harm'                  : 3,
    'S17_manic_or_hypomanic_episodes': 3,
    'S18_psychotic_symptoms'         : 3,
    'S02_catastrophizing'            : 2,
    'S04_self_blame_amplification'   : 2,
    'S05_emotional_volatility'       : 2,
    'S07_helplessness_loss_of_agency': 2,
    'S08_negative_sentiment_trend'   : 2,
    'S11_cognitive_distortion_density': 2,
    'S16_trauma_or_abuse'            : 2,
    'S03_all_or_nothing_thinking'    : 1,
    'S06_social_withdrawal_language' : 1,
    'S09_rumination_patterns'        : 1,
    'S10_emotional_numbing'          : 1,
    'S14_substance_abuse'            : 1,
}

def compute_severity_score(signal_list):
    return sum(SEVERITY_WEIGHTS.get(s, 1) for s in signal_list)

tx_df['severity_score'] = tx_df['signal_list'].apply(compute_severity_score)

top_k = tx_df.sort_values('severity_score', ascending=False).head(10)
print('=== Top 10 Most Distressed Transcripts (by Severity Score) ===')
print(top_k[['transcript_id', 'signal_list', 'signal_count', 'severity_score']].to_string(index=False))

In [ ]:
# Bar chart of top 10 severity scores
plt.figure(figsize=(12, 5))
plt.bar(
    top_k['transcript_id'],
    top_k['severity_score'],
    color='crimson',
    edgecolor='white'
)
plt.title('Top 10 Transcripts by Weighted Severity Score')
plt.xlabel('Transcript ID')
plt.ylabel('Severity Score')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 7. Signal Severity Grouping Summary

In [ ]:
HIGH_SEVERITY   = [s for s, w in SEVERITY_WEIGHTS.items() if w == 3]
MEDIUM_SEVERITY = [s for s, w in SEVERITY_WEIGHTS.items() if w == 2]
LOW_SEVERITY    = [s for s, w in SEVERITY_WEIGHTS.items() if w == 1]

def flag_severity_group(signal_list):
    if any(s in HIGH_SEVERITY for s in signal_list):
        return 'High'
    elif any(s in MEDIUM_SEVERITY for s in signal_list):
        return 'Medium'
    elif signal_list:
        return 'Low'
    else:
        return 'None'

tx_df['severity_group'] = tx_df['signal_list'].apply(flag_severity_group)

group_counts = tx_df['severity_group'].value_counts().reindex(['High', 'Medium', 'Low', 'None'])

colors = {'High': 'crimson', 'Medium': 'darkorange', 'Low': 'steelblue', 'None': 'lightgray'}

plt.figure(figsize=(8, 5))
plt.bar(
    group_counts.index,
    group_counts.values,
    color=[colors[g] for g in group_counts.index],
    edgecolor='white'
)
plt.title('Transcript Distribution by Severity Group')
plt.xlabel('Severity Group')
plt.ylabel('Number of Transcripts')
for i, (g, v) in enumerate(zip(group_counts.index, group_counts.values)):
    plt.text(i, v + 0.5, str(v), ha='center', fontweight='bold')
plt.tight_layout()
plt.show()

print('\nSummary:')
print(group_counts)

## 8. Merge Utterances with Transcript Signals

In [ ]:
# Join utterances with transcript-level signal metadata
# Useful for Week 2 model training: each utterance inherits its transcript's labels
merged_df = utt_df.merge(
    tx_df[['transcript_id', 'signal_list', 'signal_count', 'severity_score', 'severity_group']],
    on='transcript_id',
    how='left'
)

print(f'Merged shape: {merged_df.shape}')
print(f'Columns: {list(merged_df.columns)}')
merged_df.head(3)

In [ ]:
# Save merged dataset for Week 2 model training
OUTPUT_PATH = r'C:\Users\samee\Documents\CHIEAC\processed\utterances_with_signals.jsonl'

with open(OUTPUT_PATH, 'w', encoding='utf-8') as f:
    for _, row in merged_df.iterrows():
        record = {
            'transcript_id' : row['transcript_id'],
            'patient_turn_id': row['patient_turn_id'],
            'text'          : row['text'],
            'text_length'   : row['text_length'],
            'signal_list'   : row['signal_list'],
            'signal_count'  : row['signal_count'],
            'severity_score': row['severity_score'],
            'severity_group': row['severity_group']
        }
        f.write(json.dumps(record) + '\n')

print(f'Saved merged utterances to: {OUTPUT_PATH}')